In [45]:
import pandas as pd 
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.ensemble import AdaBoostClassifier

In [46]:
df = pd.read_csv('Social_Network_Ads.csv')
df.head()

,Age,EstimatedSalary,Purchased
0,19,19000,0
1,35,20000,0
2,26,43000,0
3,27,57000,0
4,19,76000,0


In [47]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 400 entries, 0 to 399
Data columns (total 3 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   Age              400 non-null    int64
 1   EstimatedSalary  400 non-null    int64
 2   Purchased        400 non-null    int64
dtypes: int64(3)
memory usage: 9.5 KB


In [48]:
df.duplicated().sum()

np.int64(33)

In [49]:
df= df.drop_duplicates().reset_index(drop=True)
print(df.duplicated().sum())
print(f'Shape of the dataset after droppoing duplicate values is ',df1.shape)

0
Shape of the dataset after droppoing duplicate values is  (367, 3)


In [50]:
df.isnull().sum()

Age                0
EstimatedSalary    0
Purchased          0
dtype: int64

In [51]:
x= df[['Age','EstimatedSalary']]
y= df['Purchased']

#### Train-test split

In [52]:
xtrain, xtest, ytrain, ytest= train_test_split(x,y, train_size=.8, random_state=13)


Splits the data into training (80%) and testing (20%) sets. random_state=13 ensures reproducibility — running this again gives the same split.

#### Defining hyperparameter grid

In [53]:
param= {'n_estimators': [50, 100, 200, 500],
    'learning_rate': [0.01, 0.1, 0.5, 1.0]
    }

 Defines the search space for GridSearchCV. n_estimators controls how many weak learners (decision stumps by default) AdaBoost trains sequentially. learning_rate controls how much each weak learner's contribution is shrunk lower values need more estimators to converge but can generalize better.

#### Setting up GridSearchCV

In [54]:
grid = GridSearchCV(
    AdaBoostClassifier(),
    param,
    cv=5,
    scoring='accuracy'
    )

Creates a GridSearchCV object that will exhaustively try all 16 hyperparameter combinations using 5-fold cross-validation, scoring each combination by accuracy. The base estimator defaults to DecisionTreeClassifier(max_depth=1) (a "decision stump") since no estimator/base_estimator is specified.

In [55]:
grid.fit(xtrain, ytrain)

print("Best Parameters:", grid.best_params_)
print("Best Score:", grid.best_score_)

Best Parameters: {'learning_rate': 1.0, 'n_estimators': 50}
Best Score: 0.9046756282875512


 Runs the full grid search fitting AdaBoost 80 times (16 combos × 5 folds) on the training data — then prints the best-performing hyperparameter combination and its average cross-validation accuracy. Again {'learning_rate': 1.0, 'n_estimators': 50} performed best. 90% is the average cross-validation accuracy on the training folds, indicating a fairly strong model for this 2-feature classification problem.

#### Taking the best model found by GridSearchCV

In [56]:
best_model = grid.best_estimator_
test_acc= best_model.score(xtest,ytest)
print('Test Accuracy',test_acc)

Test Accuracy 0.8513513513513513


Result interpretation: Test accuracy (85%) is slightly lower than the CV accuracy (90.46%), a ~5% gap that's normal for a small test set (~74 samples) roughly 4 extra misclassifications explain the whole difference. This isn't strong evidence of overfitting; it's mostly variance from a small sample. Overall, 85% is a solid result for a 2-feature model.